In [1]:
1+21

22

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

from scipy import sparse

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

# --------------------------------------------------
# Project directories
# --------------------------------------------------

project_dir = Path.cwd().parent

data_dir = project_dir / "data"
tables_dir = project_dir / "tables"
figures_dir = project_dir / "figures"

tables_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

print("Project:", project_dir)
print("Data:", data_dir)
print("Tables:", tables_dir)
print("Figures:", figures_dir)

Project: c:\Users\shafi\number-simplex-reproduction
Data: c:\Users\shafi\number-simplex-reproduction\data
Tables: c:\Users\shafi\number-simplex-reproduction\tables
Figures: c:\Users\shafi\number-simplex-reproduction\figures


In [3]:
# ============================================================
# STEP 2 — Load YFU arithmetic data
# ============================================================

subject = "YFU"

raw_dir = data_dir / "raw"
subject_dir = raw_dir / subject / "arithmetic"

spike_path = subject_dir / "spikes.mat"
behav_path = subject_dir / "photoBehavEvents.csv"

print("Spike file exists:", spike_path.exists())
print("Behavior file exists:", behav_path.exists())

Spike file exists: True
Behavior file exists: True


In [5]:
# ============================================================
# Helper — decode MATLAB strings stored as references
# ============================================================

def decode_matlab_string(f, ref):
    obj = f[ref]
    values = obj[()].flatten()
    return "".join(chr(int(x)) for x in values)

In [6]:
with h5py.File(spike_path, "r") as f:

    spikes = f["spikes"]

    yfu_spikes = sparse.csc_matrix(
        (
            spikes["data"][:],
            spikes["ir"][:],
            spikes["jc"][:]
        ),
        shape=(
            int(spikes.attrs["MATLAB_sparse"]),
            len(spikes["jc"]) - 1
        )
    )

    regions_obj = f["regionsVect"]

    yfu_regions = [
        decode_matlab_string(f, ref)
        for ref in regions_obj[0]
    ]

yfu_behav = pd.read_csv(behav_path)

In [7]:
print("Spike matrix:", yfu_spikes.shape)
print("Behavior:", yfu_behav.shape)
print("Region labels:", len(yfu_regions))

print("\nRegions:")
for region, count in zip(
    *np.unique(yfu_regions, return_counts=True)
):
    print(region, count)

Spike matrix: (78, 2273935)
Behavior: (300, 22)
Region labels: 78

Regions:
acc 16
amy 23
hpc 39


In [8]:
# ============================================================
# STEP 3A — Align operand times for YFU
# ============================================================

yfu_aligned = yfu_behav.copy()

# operationFirst == 1 means operation sign appeared first
yfu_aligned["operand1_time"] = np.where(
    yfu_aligned["operationFirst"] == 1,
    yfu_aligned["tCue2"],
    yfu_aligned["tCue1"]
)

yfu_aligned["operand2_time"] = np.where(
    yfu_aligned["operationFirst"] == 1,
    yfu_aligned["tCue3"],
    yfu_aligned["tCue2"]
)

In [9]:
# ============================================================
# STEP 3B — Pool Operand 1 + Operand 2 presentations
# ============================================================

presentation_rows = []

for trial_idx, row in yfu_aligned.iterrows():

    # Operand 1
    if (
        pd.notna(row["cue1"])
        and pd.notna(row["operand1_time"])
        and 1 <= row["cue1"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 1,
            "number": int(row["cue1"]),
            "onset_ms": row["operand1_time"]
        })

    # Operand 2
    if (
        pd.notna(row["cue2"])
        and pd.notna(row["operand2_time"])
        and 1 <= row["cue2"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 2,
            "number": int(row["cue2"]),
            "onset_ms": row["operand2_time"]
        })

yfu_presentations = pd.DataFrame(presentation_rows)

In [10]:
print("Presentation table shape:", yfu_presentations.shape)
print("Total presentations:", len(yfu_presentations))

display(yfu_presentations.head())

print("\nPresentations per number:")
print(
    yfu_presentations["number"]
    .value_counts()
    .sort_index()
)

Presentation table shape: (494, 4)
Total presentations: 494


,trial_index,operand,number,onset_ms
0,0,1,7,6660.833333
1,0,2,8,7410.800000
2,1,1,7,81376.233333
3,1,2,4,82126.233333
4,2,1,7,89626.100000



Presentations per number:
number
1    53
2    67
3    52
4    62
5    51
6    48
7    65
8    49
9    47
Name: count, dtype: int64


YFU happens to contain only two of those four.

In [11]:
# ============================================================
# STEP 4A — Select YFU MTL neurons
# ============================================================

mtl_regions = ["hpc", "amy"]

yfu_mtl_indices = [
    i for i, region in enumerate(yfu_regions)
    if region in mtl_regions
]

print("YFU MTL neurons:", len(yfu_mtl_indices))

print(
    "HPC:",
    sum(yfu_regions[i] == "hpc" for i in yfu_mtl_indices)
)

print(
    "AMY:",
    sum(yfu_regions[i] == "amy" for i in yfu_mtl_indices)
)

YFU MTL neurons: 62
HPC: 39
AMY: 23


In [12]:
# ============================================================
# STEP 4B — Population firing-rate matrix
# ============================================================

window_start = 50
window_end = 950
window_sec = 0.9

X_fr = np.zeros(
    (
        len(yfu_presentations),
        len(yfu_mtl_indices)
    )
)

y = yfu_presentations["number"].to_numpy()

for trial_pos, (_, presentation) in enumerate(
    yfu_presentations.iterrows()
):

    onset = presentation["onset_ms"]

    start = int(round(onset + window_start))
    end = int(round(onset + window_end))

    # All 62 MTL neurons simultaneously
    spike_counts = np.asarray(
        yfu_spikes[
            yfu_mtl_indices,
            start:end
        ].sum(axis=1)
    ).ravel()

    X_fr[trial_pos, :] = (
        spike_counts / window_sec
    )

In [13]:
print("X_fr shape:", X_fr.shape)
print("y shape:", y.shape)

print("\nFirst population vector:")
print(X_fr[0])

print("\nNumber for first presentation:")
print(y[0])

print("\nNaNs:", np.isnan(X_fr).sum())

X_fr shape: (494, 62)
y shape: (494,)

First population vector:
[ 1.11111111 14.44444444 13.33333333 12.22222222  0.          7.77777778
 26.66666667 14.44444444  2.22222222  6.66666667  0.         51.11111111
 32.22222222  3.33333333 10.         16.66666667  0.         23.33333333
  3.33333333  8.88888889 10.         24.44444444  0.          2.22222222
 17.77777778  4.44444444  0.         32.22222222  7.77777778  2.22222222
 11.11111111  0.          8.88888889  0.          6.66666667  7.77777778
  0.          3.33333333 21.11111111  8.88888889  5.55555556  2.22222222
  6.66666667  1.11111111  4.44444444  3.33333333  5.55555556  0.
  4.44444444 15.55555556  3.33333333  4.44444444  2.22222222  4.44444444
  0.          4.44444444  8.88888889  7.77777778  7.77777778 11.11111111
  3.33333333 15.55555556]

Number for first presentation:
7

NaNs: 0


In [14]:
# ============================================================
# STEP 5 — Population firing-rate decoding with LDA
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

# 10-fold stratified CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

fold_accuracies = []
y_true_all = []
y_pred_all = []

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X_fr, y),
    start=1
):

    X_train = X_fr[train_idx]
    X_test = X_fr[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Uniform prior because we do not want
    # frequent numbers to get an advantage
    priors = np.ones(9) / 9

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
        priors=priors
    )

    lda.fit(X_train, y_train)

    y_pred = lda.predict(X_test)

    acc = accuracy_score(
        y_test,
        y_pred
    )

    fold_accuracies.append(acc)

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

    print(
        f"Fold {fold}: "
        f"{100*acc:.2f}%"
    )

Fold 1: 6.00%
Fold 2: 8.00%
Fold 3: 14.00%
Fold 4: 12.00%
Fold 5: 10.20%
Fold 6: 4.08%
Fold 7: 14.29%
Fold 8: 14.29%
Fold 9: 18.37%
Fold 10: 14.29%


In [16]:
population_fr_accuracy = accuracy_score(
    y_true_all,
    y_pred_all
)

chance_accuracy = 1 / 9

print("\n============================")
print("YFU POPULATION FR DECODING")
print("============================")

print(
    f"Population accuracy: "
    f"{100*population_fr_accuracy:.2f}%"
)

print(
    f"Chance accuracy: "
    f"{100*chance_accuracy:.2f}%"
)

print(
    f"Mean fold accuracy: "
    f"{100*np.mean(fold_accuracies):.2f}%"
)

print(
    f"Fold SD: "
    f"{100*np.std(fold_accuracies):.2f}%"
)


YFU POPULATION FR DECODING
Population accuracy: 11.54%
Chance accuracy: 11.11%
Mean fold accuracy: 11.55%
Fold SD: 4.20%


too low, to decode even for population decoding accuracy


# Lets do temporal population decoding

In [17]:
# ============================================================
# STEP 6A — Temporal spike-count features for one neuron
# ============================================================

candidate_bins = [
    60, 75, 90, 100, 150,
    180, 225, 300, 450, 900
]

candidate_gammas = [
    0.2, 0.5, 0.8
]


def temporal_features_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    bin_ms
):
    """
    Build trial x temporal-bin spike-count matrix
    for one neuron.

    Arithmetic analysis window:
        50 ms to 950 ms after operand onset
        = 900 ms total
    """

    window_start = 50
    window_end = 950
    window_length = 900

    # All candidate bin sizes divide 900 exactly
    n_bins = window_length // bin_ms

    X = np.zeros(
        (len(presentations), n_bins),
        dtype=float
    )

    for i, (_, row) in enumerate(
        presentations.iterrows()
    ):

        onset = row["onset_ms"]

        start = int(
            round(onset + window_start)
        )

        for b in range(n_bins):

            bin_start = start + b * bin_ms
            bin_end = bin_start + bin_ms

            X[i, b] = spike_matrix[
                neuron_id,
                bin_start:bin_end
            ].sum()

    return X

In [18]:
# Paper unit 43 -> Python index 42

X_test_150 = temporal_features_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_presentations,
    neuron_id=42,
    bin_ms=150
)

print(
    "150-ms feature matrix:",
    X_test_150.shape
)

print(
    "\nFirst presentation:"
)

print(
    X_test_150[0]
)

150-ms feature matrix: (494, 6)

First presentation:
[1. 1. 0. 0. 0. 4.]


In [19]:
for bin_ms in [60, 300, 900]:

    X_tmp = temporal_features_one_neuron(
        yfu_spikes,
        yfu_presentations,
        neuron_id=42,
        bin_ms=bin_ms
    )

    print(
        f"{bin_ms:3d} ms ->",
        X_tmp.shape
    )

 60 ms -> (494, 15)
300 ms -> (494, 3)
900 ms -> (494, 1)


In [20]:
# ============================================================
# STEP 6B — Grid search for one neuron's best bin size + Gamma
# ============================================================

def temporal_grid_search_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    y,
    candidate_bins,
    candidate_gammas,
    random_state=42
):

    results = []

    # Number of CV folds allowed by smallest class
    _, class_counts = np.unique(
        y,
        return_counts=True
    )

    n_folds = min(
        10,
        int(class_counts.min())
    )

    cv = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=random_state
    )

    classes = np.unique(y)

    priors = np.ones(
        len(classes)
    ) / len(classes)

    # --------------------------------------------------------
    # Try each temporal bin size
    # --------------------------------------------------------
    for bin_ms in candidate_bins:

        X = temporal_features_one_neuron(
            spike_matrix=spike_matrix,
            presentations=presentations,
            neuron_id=neuron_id,
            bin_ms=bin_ms
        )

        # ----------------------------------------------------
        # Try each Gamma
        # ----------------------------------------------------
        for gamma in candidate_gammas:

            all_true = []
            all_pred = []

            valid_model = True

            for train_idx, test_idx in cv.split(X, y):

                X_train = X[train_idx]
                X_test = X[test_idx]

                y_train = y[train_idx]
                y_test = y[test_idx]

                # ============================================
                # Remove features with zero within-class
                # variance in the TRAINING data
                # ============================================

                keep_feature = np.zeros(
                    X_train.shape[1],
                    dtype=bool
                )

                for j in range(X_train.shape[1]):

                    class_variances = []

                    for c in classes:

                        values = X_train[
                            y_train == c,
                            j
                        ]

                        class_variances.append(
                            np.var(values)
                        )

                    # Keep if at least one class has variation
                    keep_feature[j] = (
                        np.any(
                            np.asarray(class_variances) > 0
                        )
                    )

                X_train_use = X_train[:, keep_feature]
                X_test_use = X_test[:, keep_feature]

                # No usable temporal features
                if X_train_use.shape[1] == 0:

                    valid_model = False
                    break

                # ============================================
                # Regularized LDA
                # Gamma acts as covariance shrinkage
                # ============================================

                lda = LinearDiscriminantAnalysis(
                    solver="eigen",
                    shrinkage=gamma,
                    priors=priors
                )

                try:

                    lda.fit(
                        X_train_use,
                        y_train
                    )

                    pred = lda.predict(
                        X_test_use
                    )

                except Exception:

                    valid_model = False
                    break

                all_true.extend(y_test)
                all_pred.extend(pred)

            # ------------------------------------------------
            # Accuracy pooled over all held-out predictions
            # ------------------------------------------------

            if valid_model:

                accuracy = accuracy_score(
                    all_true,
                    all_pred
                )

            else:

                accuracy = np.nan

            results.append({
                "bin_ms": bin_ms,
                "gamma": gamma,
                "n_bins": X.shape[1],
                "cv_accuracy": accuracy,
                "n_folds": n_folds
            })

    results_df = pd.DataFrame(results)

    # Highest valid CV accuracy
    valid = results_df.dropna(
        subset=["cv_accuracy"]
    )

    if len(valid) == 0:
        return results_df, None

    best_idx = valid[
        "cv_accuracy"
    ].idxmax()

    best = results_df.loc[
        best_idx
    ].to_dict()

    return results_df, best

In [21]:
grid_43, best_43 = temporal_grid_search_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_presentations,
    neuron_id=42,          # YFU.hpc.43
    y=y,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    random_state=42
)

print("BEST PARAMETERS")
print("================")

print(
    "Best bin size:",
    best_43["bin_ms"],
    "ms"
)

print(
    "Best Gamma:",
    best_43["gamma"]
)

print(
    "Number of bins:",
    best_43["n_bins"]
)

print(
    f"CV accuracy: "
    f"{100*best_43['cv_accuracy']:.2f}%"
)

BEST PARAMETERS
Best bin size: 150.0 ms
Best Gamma: 0.2
Number of bins: 6.0
CV accuracy: 14.98%


Grid search scoreboard

In [22]:
display(
    grid_43.sort_values(
        "cv_accuracy",
        ascending=False
    ).reset_index(drop=True)
)

,bin_ms,gamma,n_bins,cv_accuracy,n_folds
0,150,0.5,6,0.149798,10
1,150,0.2,6,0.149798,10
2,150,0.8,6,0.149798,10
3,75,0.5,12,0.143725,10
4,75,0.8,12,0.139676,10
5,75,0.2,12,0.137652,10
6,100,0.8,9,0.123482,10
7,90,0.8,10,0.123482,10
8,180,0.2,5,0.123482,10
9,100,0.2,9,0.121457,10


In [23]:
# ============================================================
# STEP 6C — Fit optimized temporal LDA for YFU.hpc.43
# ============================================================

best_bin = int(best_43["bin_ms"])
best_gamma = float(best_43["gamma"])

# Raw temporal-bin features for this neuron
X_43 = temporal_features_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_presentations,
    neuron_id=42,
    bin_ms=best_bin
)

print("Raw temporal feature shape:", X_43.shape)

Raw temporal feature shape: (494, 6)


In [24]:
classes = np.unique(y)
priors = np.ones(len(classes)) / len(classes)

lda_43 = LinearDiscriminantAnalysis(
    solver="eigen",
    shrinkage=best_gamma,
    priors=priors
)

lda_43.fit(X_43, y)

,"solver solver: {'svd', 'lsqr', 'eigen'}, default='svd'Solver to use, possible values: - 'svd': Singular value decomposition (default). Does not compute the covariance matrix, therefore this solver is recommended for data with a large number of features. - 'lsqr': Least squares solution. Can be combined with shrinkage or custom covariance estimator. - 'eigen': Eigenvalue decomposition. Can be combined with shrinkage or custom covariance estimator... versionchanged:: 1.2 `solver=""svd""` now has experimental Array API support. See the :ref:`Array API User Guide <array_api>` for more details.",'eigen'
,"shrinkage shrinkage: 'auto' or float, default=NoneShrinkage parameter, possible values: - None: no shrinkage (default). - 'auto': automatic shrinkage using the Ledoit-Wolf lemma. - float between 0 and 1: fixed shrinkage parameter.This should be left to None if `covariance_estimator` is used.Note that shrinkage works only with 'lsqr' and 'eigen' solvers.For a usage example, see:ref:`sphx_glr_auto_examples_classification_plot_lda.py`.",0.2
,"priors priors: array-like of shape (n_classes,), default=NoneThe class prior probabilities. By default, the class proportions areinferred from the training data.","array([0.1111..., 0.11111111])"
,"n_components n_components: int, default=NoneNumber of components (<= min(n_classes - 1, n_features)) fordimensionality reduction. If None, will be set tomin(n_classes - 1, n_features). This parameter only affects the`transform` method.For a usage example, see:ref:`sphx_glr_auto_examples_decomposition_plot_pca_vs_lda.py`.",None
,"store_covariance store_covariance: bool, default=FalseIf True, explicitly compute the weighted within-class covariancematrix when solver is 'svd'. The matrix is always computedand stored for the other solvers... versionadded:: 0.17",False
,"tol tol: float, default=1.0e-4Absolute threshold for a singular value of X to be consideredsignificant, used to estimate the rank of X. Dimensions whosesingular values are non-significant are discarded. Only used ifsolver is 'svd'... versionadded:: 0.17",0.0001
,"covariance_estimator covariance_estimator: covariance estimator, default=NoneIf not None, `covariance_estimator` is used to estimatethe covariance matrices instead of relying on the empiricalcovariance estimator (with potential shrinkage).The object should have a fit method and a ``covariance_`` attributelike the estimators in :mod:`sklearn.covariance`.if None the shrinkage parameter drives the estimate.This should be left to None if `shrinkage` is used.Note that `covariance_estimator` works only with 'lsqr' and 'eigen'solvers... versionadded:: 0.24",None
Name,Type,Value
"classes_ classes_: array-like of shape (n_classes,)Unique class labels.","ndarray[int64](9,)","[1,2,3,...,7,8,9]"
"coef_ coef_: ndarray of shape (n_features,) or (n_classes, n_features)Weight vector(s).","ndarray[float64](9, 6)","[[0.98,1.22,0.92,0.77,0.87,1.22], [1.13,1.05,1.16,1.15,0.86,0.92], [1.39,0.92,0.95,1.18,0.9 ,1.46], ..., [0.92,0.6 ,1.02,0.96,1.08,1.02], [1.15,0.97,1.02,0.85,0.92,0.79], [1.26,1.04,0.94,1.01,1.42,0.83]]"
"covariance_ covariance_: array-like of shape (n_features, n_features)Weighted within-class covariance matrix. It corresponds to`sum_k prior_k * C_k` where `C_k` is the covariance matrix of thesamples in class `k`. The `C_k` are estimated using the (potentiallyshrunk) biased estimator of covariance. If solver is 'svd', onlyexists when `store_covariance` is True.","ndarray[float64](6, 6)","[[ 1.04,-0.02, 0.01,-0.01,-0.03,-0.04], [-0.02, 0.91, 0.04, 0.05, 0.04,-0.02], [ 0.01, 0.04, 0.94,-0.03,-0.03, 0.05], [-0.01, 0.05,-0.03, 0.87, 0.03, 0. ], [-0.03, 0.04,-0.03, 0.03, 1.06, 0.01], [-0.04,-0.02, 0.05, 0. , 0.01, 0.93]]"


In [25]:
TC_43_all = lda_43.transform(X_43)

print("All LDA component shape:", TC_43_all.shape)

All LDA component shape: (494, 6)


In [26]:
n_keep = min(
    3,
    TC_43_all.shape[1]
)

TC_43 = TC_43_all[:, :n_keep]

print("Retained TC shape:", TC_43.shape)

print("\nFirst presentation:")
print(TC_43[0])

Retained TC shape: (494, 3)

First presentation:
[-4.17473037 -1.31330623  0.58018387]


---

Now concanete all 3TC x  62 neurons = 186 poulation features 

So for each outer CV fold:

$$ \text{training presentations} \rightarrow \text{choose best bin + }\Gamma \rightarrow \text{fit temporal LDA} $$

and only then:

transform held-out test presentations.

In [27]:
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
import numpy as np


def fit_temporal_block_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    y,
    outer_train_idx,
    outer_test_idx,
    candidate_bins,
    candidate_gammas,
    random_state=42,
    max_components=3
):
    """
    For ONE neuron and ONE outer CV fold:

    1. Use outer-training trials only.
    2. Inner CV chooses best bin size + Gamma.
    3. Fit optimized LDA on all outer-training trials.
    4. Transform both outer train and held-out outer test trials.
    5. Keep up to first 3 temporal components.

    Returns
    -------
    TC_train
    TC_test
    info
    """

    y_train_outer = y[outer_train_idx]

    classes, counts = np.unique(
        y_train_outer,
        return_counts=True
    )

    n_inner_folds = min(
        10,
        int(counts.min())
    )

    inner_cv = StratifiedKFold(
        n_splits=n_inner_folds,
        shuffle=True,
        random_state=random_state
    )

    priors = np.ones(len(classes)) / len(classes)

    best_accuracy = -np.inf
    best_bin = None
    best_gamma = None

    # =====================================================
    # INNER GRID SEARCH
    # =====================================================

    for bin_ms in candidate_bins:

        # Build features for all presentations once
        X_all = temporal_features_one_neuron(
            spike_matrix=spike_matrix,
            presentations=presentations,
            neuron_id=neuron_id,
            bin_ms=bin_ms
        )

        X_outer_train = X_all[outer_train_idx]

        for gamma in candidate_gammas:

            inner_true = []
            inner_pred = []

            valid = True

            for inner_train_rel, inner_val_rel in inner_cv.split(
                X_outer_train,
                y_train_outer
            ):

                X_inner_train = X_outer_train[
                    inner_train_rel
                ]

                X_inner_val = X_outer_train[
                    inner_val_rel
                ]

                y_inner_train = y_train_outer[
                    inner_train_rel
                ]

                y_inner_val = y_train_outer[
                    inner_val_rel
                ]

                # -----------------------------------------
                # Remove unusable temporal bins
                # using TRAINING data only
                # -----------------------------------------

                keep = np.zeros(
                    X_inner_train.shape[1],
                    dtype=bool
                )

                for j in range(
                    X_inner_train.shape[1]
                ):

                    variances = []

                    for c in classes:

                        values = X_inner_train[
                            y_inner_train == c,
                            j
                        ]

                        variances.append(
                            np.var(values)
                        )

                    keep[j] = np.any(
                        np.asarray(variances) > 0
                    )

                if keep.sum() == 0:
                    valid = False
                    break

                lda = LinearDiscriminantAnalysis(
                    solver="eigen",
                    shrinkage=gamma,
                    priors=priors
                )

                try:

                    lda.fit(
                        X_inner_train[:, keep],
                        y_inner_train
                    )

                    pred = lda.predict(
                        X_inner_val[:, keep]
                    )

                except Exception:

                    valid = False
                    break

                inner_true.extend(
                    y_inner_val
                )

                inner_pred.extend(
                    pred
                )

            if not valid:
                continue

            acc = accuracy_score(
                inner_true,
                inner_pred
            )

            if acc > best_accuracy:

                best_accuracy = acc
                best_bin = bin_ms
                best_gamma = gamma

    # =====================================================
    # FIT THE WINNING TEMPORAL MODEL ON ALL OUTER TRAINING
    # =====================================================

    if best_bin is None:

        return None, None, {
            "valid": False
        }

    X_best = temporal_features_one_neuron(
        spike_matrix=spike_matrix,
        presentations=presentations,
        neuron_id=neuron_id,
        bin_ms=best_bin
    )

    X_train = X_best[
        outer_train_idx
    ]

    X_test = X_best[
        outer_test_idx
    ]

    # Determine usable temporal bins again
    # from the COMPLETE OUTER TRAINING SET only

    keep = np.zeros(
        X_train.shape[1],
        dtype=bool
    )

    for j in range(
        X_train.shape[1]
    ):

        variances = []

        for c in classes:

            values = X_train[
                y_train_outer == c,
                j
            ]

            variances.append(
                np.var(values)
            )

        keep[j] = np.any(
            np.asarray(variances) > 0
        )

    if keep.sum() == 0:

        return None, None, {
            "valid": False
        }

    final_lda = LinearDiscriminantAnalysis(
        solver="eigen",
        shrinkage=best_gamma,
        priors=priors
    )

    final_lda.fit(
        X_train[:, keep],
        y_train_outer
    )

    TC_train_all = final_lda.transform(
        X_train[:, keep]
    )

    TC_test_all = final_lda.transform(
        X_test[:, keep]
    )

    n_keep = min(
        max_components,
        TC_train_all.shape[1]
    )

    TC_train = TC_train_all[
        :, :n_keep
    ]

    TC_test = TC_test_all[
        :, :n_keep
    ]

    info = {
        "valid": True,
        "neuron_id": neuron_id,
        "best_bin": best_bin,
        "best_gamma": best_gamma,
        "inner_accuracy": best_accuracy,
        "n_raw_bins": X_best.shape[1],
        "n_components": n_keep
    }

    return TC_train, TC_test, info

In [28]:
outer_cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

outer_train_idx, outer_test_idx = next(
    outer_cv.split(
        np.zeros(len(y)),
        y
    )
)

In [29]:
TC_train_43, TC_test_43, info_43 = (
    fit_temporal_block_one_neuron(
        spike_matrix=yfu_spikes,
        presentations=yfu_presentations,
        neuron_id=42,
        y=y,
        outer_train_idx=outer_train_idx,
        outer_test_idx=outer_test_idx,
        candidate_bins=candidate_bins,
        candidate_gammas=candidate_gammas,
        random_state=42,
        max_components=3
    )
)

print(info_43)

print(
    "TC train shape:",
    TC_train_43.shape
)

print(
    "TC test shape:",
    TC_test_43.shape
)

{'valid': True, 'neuron_id': 42, 'best_bin': 180, 'best_gamma': 0.8, 'inner_accuracy': 0.1463963963963964, 'n_raw_bins': 5, 'n_components': 3}
TC train shape: (444, 3)
TC test shape: (50, 3)


this is on outer training fold,

Then the grid search sees only those 444 training presentations. On that subset, the best parameters happened to be: this ,

---

build one population TC matrix

In [30]:
# ============================================================
# STEP 7C — Build temporal population matrix for ONE outer fold
# ============================================================

TC_train_blocks = []
TC_test_blocks = []

neuron_info = []

for k, neuron_id in enumerate(yfu_mtl_indices, start=1):

    TC_train, TC_test, info = (
        fit_temporal_block_one_neuron(
            spike_matrix=yfu_spikes,
            presentations=yfu_presentations,
            neuron_id=neuron_id,
            y=y,
            outer_train_idx=outer_train_idx,
            outer_test_idx=outer_test_idx,
            candidate_bins=candidate_bins,
            candidate_gammas=candidate_gammas,
            random_state=42,
            max_components=3
        )
    )

    # Skip neurons that fail to produce usable temporal features
    if info["valid"]:

        TC_train_blocks.append(TC_train)
        TC_test_blocks.append(TC_test)

        info["region"] = yfu_regions[neuron_id]
        neuron_info.append(info)

    print(
        f"{k}/{len(yfu_mtl_indices)} neurons processed",
        end="\r"
    )

print("\nDone.")

62/62 neurons processed
Done.


In [31]:
# ============================================================
# Concatenate neurons into population representation
# ============================================================

X_pop_TC_train = np.hstack(
    TC_train_blocks
)

X_pop_TC_test = np.hstack(
    TC_test_blocks
)

print(
    "Population TC train shape:",
    X_pop_TC_train.shape
)

print(
    "Population TC test shape:",
    X_pop_TC_test.shape
)

print(
    "Valid neurons:",
    len(neuron_info)
)

print(
    "Total population TC features:",
    X_pop_TC_train.shape[1]
)

Population TC train shape: (444, 165)
Population TC test shape: (50, 165)
Valid neurons: 62
Total population TC features: 165


In [32]:
# ============================================================
# Concatenate neurons into population representation
# ============================================================

X_pop_TC_train = np.hstack(
    TC_train_blocks
)

X_pop_TC_test = np.hstack(
    TC_test_blocks
)

print(
    "Population TC train shape:",
    X_pop_TC_train.shape
)

print(
    "Population TC test shape:",
    X_pop_TC_test.shape
)

print(
    "Valid neurons:",
    len(neuron_info)
)

print(
    "Total population TC features:",
    X_pop_TC_train.shape[1]
)

Population TC train shape: (444, 165)
Population TC test shape: (50, 165)
Valid neurons: 62
Total population TC features: 165


In [33]:
neuron_info_df = pd.DataFrame(neuron_info)

display(
    neuron_info_df[
        [
            "neuron_id",
            "region",
            "best_bin",
            "best_gamma",
            "inner_accuracy",
            "n_raw_bins",
            "n_components"
        ]
    ].head(10)
)

,neuron_id,region,best_bin,best_gamma,inner_accuracy,n_raw_bins,n_components
0,0,amy,75,0.5,0.132883,12,3
1,1,amy,60,0.2,0.126126,15,3
2,2,amy,150,0.8,0.141892,6,3
3,3,amy,450,0.5,0.139640,2,2
4,4,amy,90,0.5,0.123874,10,3
5,5,amy,60,0.2,0.128378,15,3
6,6,amy,75,0.5,0.103604,12,3
7,7,amy,300,0.5,0.132883,3,3
8,8,amy,90,0.5,0.117117,10,3
9,9,amy,300,0.5,0.117117,3,3


In [34]:
print("\nSelected bin sizes:")
print(
    neuron_info_df["best_bin"]
    .value_counts()
    .sort_index()
)


Selected bin sizes:
best_bin
60     13
75      5
90      6
100     4
150     5
180     8
225     2
300     6
450     5
900     8
Name: count, dtype: int64


Why 165 and not \(62\times3=186\)? Because not every neuron contributed three components

In [35]:
# ============================================================
# STEP 7D — Decode held-out numbers from population TCs
# ============================================================

y_train_outer = y[outer_train_idx]
y_test_outer = y[outer_test_idx]

classes = np.unique(y_train_outer)
priors = np.ones(len(classes)) / len(classes)

population_lda = LinearDiscriminantAnalysis(
    solver="lsqr",
    shrinkage="auto",
    priors=priors
)

population_lda.fit(
    X_pop_TC_train,
    y_train_outer
)

y_pred_outer = population_lda.predict(
    X_pop_TC_test
)

outer_accuracy = accuracy_score(
    y_test_outer,
    y_pred_outer
)

print(
    f"Outer-fold temporal population accuracy: "
    f"{100*outer_accuracy:.2f}%"
)

print(
    f"Chance: "
    f"{100/9:.2f}%"
)

Outer-fold temporal population accuracy: 8.00%
Chance: 11.11%


this is just one fold

In [36]:
# ============================================================
# STEP 8A — Precompute temporal features for YFU
# ============================================================

temporal_cache = {}

for k, neuron_id in enumerate(yfu_mtl_indices, start=1):

    temporal_cache[neuron_id] = {}

    for bin_ms in candidate_bins:

        X_tmp = temporal_features_one_neuron(
            spike_matrix=yfu_spikes,
            presentations=yfu_presentations,
            neuron_id=neuron_id,
            bin_ms=bin_ms
        )

        temporal_cache[neuron_id][bin_ms] = X_tmp

    print(
        f"{k}/{len(yfu_mtl_indices)} neurons cached",
        end="\r"
    )

print("\nTemporal feature cache complete.")

62/62 neurons cached
Temporal feature cache complete.


In [37]:
print(
    temporal_cache[42][150].shape
)

print(
    temporal_cache[42][60].shape
)

(494, 6)
(494, 15)


In [39]:
def fit_temporal_block_cached(
    temporal_cache,
    neuron_id,
    y,
    outer_train_idx,
    outer_test_idx,
    candidate_bins,
    candidate_gammas,
    random_state=42,
    max_components=3
):

    y_outer_train = y[outer_train_idx]

    classes, counts = np.unique(
        y_outer_train,
        return_counts=True
    )

    n_inner_folds = min(
        10,
        int(counts.min())
    )

    inner_cv = StratifiedKFold(
        n_splits=n_inner_folds,
        shuffle=True,
        random_state=random_state
    )

    priors = np.ones(
        len(classes)
    ) / len(classes)

    best_accuracy = -np.inf
    best_bin = None
    best_gamma = None

    # =====================================================
    # INNER GRID SEARCH
    # =====================================================

    for bin_ms in candidate_bins:

        X_all = temporal_cache[
            neuron_id
        ][bin_ms]

        X_outer_train = X_all[
            outer_train_idx
        ]

        for gamma in candidate_gammas:

            y_true_inner = []
            y_pred_inner = []

            valid = True

            for train_rel, val_rel in inner_cv.split(
                X_outer_train,
                y_outer_train
            ):

                X_inner_train = X_outer_train[
                    train_rel
                ]

                X_inner_val = X_outer_train[
                    val_rel
                ]

                y_inner_train = y_outer_train[
                    train_rel
                ]

                y_inner_val = y_outer_train[
                    val_rel
                ]

                keep = np.zeros(
                    X_inner_train.shape[1],
                    dtype=bool
                )

                for j in range(
                    X_inner_train.shape[1]
                ):

                    variances = [
                        np.var(
                            X_inner_train[
                                y_inner_train == c,
                                j
                            ]
                        )
                        for c in classes
                    ]

                    keep[j] = np.any(
                        np.asarray(variances) > 0
                    )

                if keep.sum() == 0:
                    valid = False
                    break

                lda = LinearDiscriminantAnalysis(
                    solver="eigen",
                    shrinkage=gamma,
                    priors=priors
                )

                try:

                    lda.fit(
                        X_inner_train[:, keep],
                        y_inner_train
                    )

                    pred = lda.predict(
                        X_inner_val[:, keep]
                    )

                except Exception:

                    valid = False
                    break

                y_true_inner.extend(
                    y_inner_val
                )

                y_pred_inner.extend(
                    pred
                )

            if not valid:
                continue

            acc = accuracy_score(
                y_true_inner,
                y_pred_inner
            )

            if acc > best_accuracy:

                best_accuracy = acc
                best_bin = bin_ms
                best_gamma = gamma

    if best_bin is None:

        return None, None, {
            "valid": False
        }

    # =====================================================
    # FIT OPTIMAL MODEL ON FULL OUTER TRAINING SET
    # =====================================================

    X_best = temporal_cache[
        neuron_id
    ][best_bin]

    X_train = X_best[
        outer_train_idx
    ]

    X_test = X_best[
        outer_test_idx
    ]

    keep = np.zeros(
        X_train.shape[1],
        dtype=bool
    )

    for j in range(
        X_train.shape[1]
    ):

        variances = [
            np.var(
                X_train[
                    y_outer_train == c,
                    j
                ]
            )
            for c in classes
        ]

        keep[j] = np.any(
            np.asarray(variances) > 0
        )

    if keep.sum() == 0:

        return None, None, {
            "valid": False
        }

    final_lda = LinearDiscriminantAnalysis(
        solver="eigen",
        shrinkage=best_gamma,
        priors=priors
    )

    final_lda.fit(
        X_train[:, keep],
        y_outer_train
    )

    TC_train_all = final_lda.transform(
        X_train[:, keep]
    )

    TC_test_all = final_lda.transform(
        X_test[:, keep]
    )

    n_keep = min(
        max_components,
        TC_train_all.shape[1]
    )

    return (
        TC_train_all[:, :n_keep],
        TC_test_all[:, :n_keep],
        {
            "valid": True,
            "neuron_id": neuron_id,
            "best_bin": best_bin,
            "best_gamma": best_gamma,
            "inner_accuracy": best_accuracy,
            "n_components": n_keep
        }
    )

In [40]:
# ============================================================
# STEP 8B — Full nested-CV temporal population decoding
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

all_true = []
all_pred = []

fold_results = []

for fold, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(
        np.zeros(len(y)),
        y
    ),
    start=1
):

    print(
        f"\n===== OUTER FOLD {fold}/10 ====="
    )

    train_blocks = []
    test_blocks = []

    fold_neuron_info = []

    for k, neuron_id in enumerate(
        yfu_mtl_indices,
        start=1
    ):

        TC_train, TC_test, info = (
            fit_temporal_block_cached(
                temporal_cache=temporal_cache,
                neuron_id=neuron_id,
                y=y,
                outer_train_idx=outer_train_idx,
                outer_test_idx=outer_test_idx,
                candidate_bins=candidate_bins,
                candidate_gammas=candidate_gammas,
                random_state=42,
                max_components=3
            )
        )

        if info["valid"]:

            train_blocks.append(
                TC_train
            )

            test_blocks.append(
                TC_test
            )

            fold_neuron_info.append(
                info
            )

        print(
            f"{k}/{len(yfu_mtl_indices)} neurons",
            end="\r"
        )

    X_train_pop = np.hstack(
        train_blocks
    )

    X_test_pop = np.hstack(
        test_blocks
    )

    y_train = y[
        outer_train_idx
    ]

    y_test = y[
        outer_test_idx
    ]

    priors = np.ones(9) / 9

    population_lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
        priors=priors
    )

    population_lda.fit(
        X_train_pop,
        y_train
    )

    y_pred = population_lda.predict(
        X_test_pop
    )

    fold_accuracy = accuracy_score(
        y_test,
        y_pred
    )

    all_true.extend(
        y_test
    )

    all_pred.extend(
        y_pred
    )

    fold_results.append({
        "fold": fold,
        "accuracy": fold_accuracy,
        "n_features":
            X_train_pop.shape[1],
        "n_valid_neurons":
            len(fold_neuron_info)
    })

    print(
        f"\nFold {fold} accuracy: "
        f"{100*fold_accuracy:.2f}%"
    )


===== OUTER FOLD 1/10 =====
62/62 neurons
Fold 1 accuracy: 8.00%

===== OUTER FOLD 2/10 =====
62/62 neurons
Fold 2 accuracy: 12.00%

===== OUTER FOLD 3/10 =====
62/62 neurons
Fold 3 accuracy: 10.00%

===== OUTER FOLD 4/10 =====
62/62 neurons
Fold 4 accuracy: 14.00%

===== OUTER FOLD 5/10 =====
62/62 neurons
Fold 5 accuracy: 14.29%

===== OUTER FOLD 6/10 =====
62/62 neurons
Fold 6 accuracy: 12.24%

===== OUTER FOLD 7/10 =====
62/62 neurons
Fold 7 accuracy: 12.24%

===== OUTER FOLD 8/10 =====
62/62 neurons
Fold 8 accuracy: 4.08%

===== OUTER FOLD 9/10 =====
62/62 neurons
Fold 9 accuracy: 18.37%

===== OUTER FOLD 10/10 =====
62/62 neurons
Fold 10 accuracy: 8.16%


In [41]:
population_tc_accuracy = accuracy_score(
    all_true,
    all_pred
)

fold_results_df = pd.DataFrame(
    fold_results
)

print("\n=================================")
print("YFU POPULATION TEMPORAL DECODING")
print("=================================")

print(
    f"Population temporal accuracy: "
    f"{100*population_tc_accuracy:.2f}%"
)

print(
    f"Chance: "
    f"{100/9:.2f}%"
)

print(
    f"Mean fold accuracy: "
    f"{100*fold_results_df['accuracy'].mean():.2f}%"
)

print(
    f"Fold SD: "
    f"{100*fold_results_df['accuracy'].std(ddof=0):.2f}%"
)

display(fold_results_df)


YFU POPULATION TEMPORAL DECODING
Population temporal accuracy: 11.34%
Chance: 11.11%
Mean fold accuracy: 11.34%
Fold SD: 3.78%


,fold,accuracy,n_features,n_valid_neurons
0,1,0.080000,165,62
1,2,0.120000,166,62
2,3,0.100000,166,62
3,4,0.140000,168,62
4,5,0.142857,169,62
5,6,0.122449,174,62
6,7,0.122449,169,62
7,8,0.040816,165,62
8,9,0.183673,171,62
9,10,0.081633,164,62


Can an individual neuron’s temporal response carry significant number information?  vs

Can we predict the exact number on a new individual presentation from the whole population?


---

oPERAND SPEFICITY

In [42]:
# ============================================================
# OPERAND SPECIFICITY — STEP 1
# Split pooled YFU presentations into Operand 1 and Operand 2
# ============================================================

yfu_op1 = yfu_presentations[
    yfu_presentations["operand"] == 1
].copy()

yfu_op2 = yfu_presentations[
    yfu_presentations["operand"] == 2
].copy()

print("Operand 1 presentations:", len(yfu_op1))
print("Operand 2 presentations:", len(yfu_op2))

print("\nOperand 1 number counts:")
print(
    yfu_op1["number"]
    .value_counts()
    .sort_index()
)

print("\nOperand 2 number counts:")
print(
    yfu_op2["number"]
    .value_counts()
    .sort_index()
)

Operand 1 presentations: 244
Operand 2 presentations: 250

Operand 1 number counts:
number
1    28
2    30
3    27
4    27
5    23
6    26
7    37
8    24
9    22
Name: count, dtype: int64

Operand 2 number counts:
number
1    25
2    37
3    25
4    35
5    28
6    22
7    28
8    25
9    25
Name: count, dtype: int64


In [44]:
# ============================================================
# OPERAND SPECIFICITY — STEP 2
# YFU.hpc.43: Operand 1 vs Operand 2 firing-rate decoding
# ============================================================

neuron_id = 42   # YFU.hpc.43, Python index

result_op1_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=neuron_id
)

result_op2_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=neuron_id
)

print("YFU.hpc.43")
print("==============================")

print("\nOPERAND 1")
print(result_op1_fr)

print("\nOPERAND 2")
print(result_op2_fr)

NameError: name 'decode_fr_one_neuron' is not defined

In [46]:
# ============================================================
# OPERAND SPECIFICITY — FR decoder for one neuron
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
import numpy as np


def decode_fr_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    random_state=42
):
    """
    9-way number decoding using ONE firing-rate feature
    from one neuron.
    """

    # --------------------------------------------
    # Build firing-rate feature
    # 50–950 ms = 900 ms window
    # --------------------------------------------

    X = np.zeros((len(presentations), 1))

    for i, (_, row) in enumerate(presentations.iterrows()):

        onset = row["onset_ms"]

        start = int(round(onset + 50))
        end   = int(round(onset + 950))

        spike_count = spike_matrix[
            neuron_id,
            start:end
        ].sum()

        # firing rate in Hz
        X[i, 0] = spike_count / 0.9

    y_local = presentations[
        "number"
    ].to_numpy().astype(int)

    # --------------------------------------------
    # 10-fold stratified CV
    # --------------------------------------------

    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=random_state
    )

    classes = np.unique(y_local)

    priors = np.ones(
        len(classes)
    ) / len(classes)

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in cv.split(X, y_local):

        lda = LinearDiscriminantAnalysis(
            priors=priors
        )

        try:

            lda.fit(
                X[train_idx],
                y_local[train_idx]
            )

            pred = lda.predict(
                X[test_idx]
            )

        except Exception:

            continue

        y_true_all.extend(
            y_local[test_idx]
        )

        y_pred_all.extend(
            pred
        )

    accuracy = accuracy_score(
        y_true_all,
        y_pred_all
    )

    return {
        "neuron_id": neuron_id,
        "n_presentations": len(y_local),
        "accuracy": accuracy,
        "chance": 1 / len(classes)
    }

In [47]:
# ============================================================
# OPERAND SPECIFICITY — STEP 2
# YFU.hpc.43: Operand 1 vs Operand 2 firing-rate decoding
# ============================================================

neuron_id = 42   # YFU.hpc.43, Python index

result_op1_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=neuron_id
)

result_op2_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=neuron_id
)

print("YFU.hpc.43")
print("==============================")

print("\nOPERAND 1")
print(result_op1_fr)

print("\nOPERAND 2")
print(result_op2_fr)

YFU.hpc.43

OPERAND 1
{'neuron_id': 42, 'n_presentations': 244, 'accuracy': 0.08196721311475409, 'chance': 0.1111111111111111}

OPERAND 2
{'neuron_id': 42, 'n_presentations': 250, 'accuracy': 0.1, 'chance': 0.1111111111111111}


In [48]:
# ============================================================
# OPERAND SPECIFICITY — FR decoder for one neuron
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
import numpy as np


def decode_fr_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    random_state=42
):
    """
    9-way number decoding using ONE firing-rate feature
    from one neuron.
    """

    # --------------------------------------------
    # Build firing-rate feature
    # 50–950 ms = 900 ms window
    # --------------------------------------------

    X = np.zeros((len(presentations), 1))

    for i, (_, row) in enumerate(presentations.iterrows()):

        onset = row["onset_ms"]

        start = int(round(onset + 50))
        end   = int(round(onset + 950))

        spike_count = spike_matrix[
            neuron_id,
            start:end
        ].sum()

        # firing rate in Hz
        X[i, 0] = spike_count / 0.9

    y_local = presentations[
        "number"
    ].to_numpy().astype(int)

    # --------------------------------------------
    # 10-fold stratified CV
    # --------------------------------------------

    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=random_state
    )

    classes = np.unique(y_local)

    priors = np.ones(
        len(classes)
    ) / len(classes)

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in cv.split(X, y_local):

        lda = LinearDiscriminantAnalysis(
            priors=priors
        )

        try:

            lda.fit(
                X[train_idx],
                y_local[train_idx]
            )

            pred = lda.predict(
                X[test_idx]
            )

        except Exception:

            continue

        y_true_all.extend(
            y_local[test_idx]
        )

        y_pred_all.extend(
            pred
        )

    accuracy = accuracy_score(
        y_true_all,
        y_pred_all
    )

    return {
        "neuron_id": neuron_id,
        "n_presentations": len(y_local),
        "accuracy": accuracy,
        "chance": 1 / len(classes)
    }

In [49]:
neuron_id = 42   # YFU.hpc.43

result_op1_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=neuron_id
)

result_op2_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=neuron_id
)

print("YFU.hpc.43")
print("==========================")

print(
    f"Operand 1: "
    f"{100*result_op1_fr['accuracy']:.2f}% "
    f"(n={result_op1_fr['n_presentations']})"
)

print(
    f"Operand 2: "
    f"{100*result_op2_fr['accuracy']:.2f}% "
    f"(n={result_op2_fr['n_presentations']})"
)

print(
    f"Chance: "
    f"{100*result_op1_fr['chance']:.2f}%"
)

YFU.hpc.43
Operand 1: 8.20% (n=244)
Operand 2: 10.00% (n=250)
Chance: 11.11%


In [50]:
neuron_id = 42   # YFU.hpc.43

result_op1_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=neuron_id
)

result_op2_fr = decode_fr_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=neuron_id
)

print("YFU.hpc.43")
print("==========================")

print(
    f"Operand 1: "
    f"{100*result_op1_fr['accuracy']:.2f}% "
    f"(n={result_op1_fr['n_presentations']})"
)

print(
    f"Operand 2: "
    f"{100*result_op2_fr['accuracy']:.2f}% "
    f"(n={result_op2_fr['n_presentations']})"
)

print(
    f"Chance: "
    f"{100*result_op1_fr['chance']:.2f}%"
)

YFU.hpc.43
Operand 1: 8.20% (n=244)
Operand 2: 10.00% (n=250)
Chance: 11.11%


temporal_grid_search_one_neuron

In [51]:
# ============================================================
# OPERAND SPECIFICITY — STEP 3
# Temporal decoding: Operand 1 vs Operand 2
# YFU.hpc.43
# ============================================================

y_op1 = yfu_op1["number"].to_numpy().astype(int)
y_op2 = yfu_op2["number"].to_numpy().astype(int)

# -------------------------
# Operand 1
# -------------------------

grid_op1, best_op1 = temporal_grid_search_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=42,
    y=y_op1,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    random_state=42
)

# -------------------------
# Operand 2
# -------------------------

grid_op2, best_op2 = temporal_grid_search_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=42,
    y=y_op2,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    random_state=42
)

print("YFU.hpc.43 TEMPORAL DECODING")
print("================================")

print("\nOPERAND 1")
print(
    f"Best bin: {best_op1['bin_ms']:.0f} ms"
)
print(
    f"Best Gamma: {best_op1['gamma']}"
)
print(
    f"Best accuracy: "
    f"{100*best_op1['cv_accuracy']:.2f}%"
)

print("\nOPERAND 2")
print(
    f"Best bin: {best_op2['bin_ms']:.0f} ms"
)
print(
    f"Best Gamma: {best_op2['gamma']}"
)
print(
    f"Best accuracy: "
    f"{100*best_op2['cv_accuracy']:.2f}%"
)

print("\nChance: 11.11%")

YFU.hpc.43 TEMPORAL DECODING

OPERAND 1
Best bin: 150 ms
Best Gamma: 0.2
Best accuracy: 11.48%

OPERAND 2
Best bin: 150 ms
Best Gamma: 0.8
Best accuracy: 14.00%

Chance: 11.11%


Operand 2 may carry stronger temporal information for this neuron

In [52]:
# ============================================================
# OPERAND SPECIFICITY — STEP 4
# Permutation test for optimized temporal decoding
# ============================================================

def permutation_test_temporal(
    spike_matrix,
    presentations,
    neuron_id,
    y,
    candidate_bins,
    candidate_gammas,
    n_shuffles=200,
    random_state=42
):
    """
    Permutation test for optimized temporal decoding.

    For the real labels:
        run complete bin-size x Gamma grid search.

    For every shuffle:
        shuffle number labels
        repeat the COMPLETE grid search
        save the best CV accuracy.

    This produces a null distribution that includes
    the parameter-selection step.
    """

    rng = np.random.default_rng(random_state)

    # --------------------------------------------------------
    # REAL DATA
    # --------------------------------------------------------

    real_grid, real_best = temporal_grid_search_one_neuron(
        spike_matrix=spike_matrix,
        presentations=presentations,
        neuron_id=neuron_id,
        y=y,
        candidate_bins=candidate_bins,
        candidate_gammas=candidate_gammas,
        random_state=random_state
    )

    real_accuracy = real_best["cv_accuracy"]

    # --------------------------------------------------------
    # SHUFFLED DATA
    # --------------------------------------------------------

    shuffled_best_accuracies = []

    for s in range(n_shuffles):

        y_shuffle = rng.permutation(y)

        _, shuffled_best = temporal_grid_search_one_neuron(
            spike_matrix=spike_matrix,
            presentations=presentations,
            neuron_id=neuron_id,
            y=y_shuffle,
            candidate_bins=candidate_bins,
            candidate_gammas=candidate_gammas,
            random_state=random_state
        )

        if shuffled_best is not None:

            shuffled_best_accuracies.append(
                shuffled_best["cv_accuracy"]
            )

        if (s + 1) % 10 == 0:
            print(
                f"{s + 1}/{n_shuffles} shuffles complete",
                end="\r"
            )

    shuffled_best_accuracies = np.asarray(
        shuffled_best_accuracies
    )

    # --------------------------------------------------------
    # Empirical one-sided permutation p-value
    # --------------------------------------------------------

    p_value = (
        1
        + np.sum(
            shuffled_best_accuracies >= real_accuracy
        )
    ) / (
        1 + len(shuffled_best_accuracies)
    )

    return {
        "real_accuracy": real_accuracy,
        "best_bin": real_best["bin_ms"],
        "best_gamma": real_best["gamma"],
        "p_value": p_value,
        "null_mean":
            shuffled_best_accuracies.mean(),
        "null_sd":
            shuffled_best_accuracies.std(ddof=1),
        "null_95":
            np.percentile(
                shuffled_best_accuracies,
                95
            ),
        "shuffled_accuracies":
            shuffled_best_accuracies
    }

In [53]:
print("Running Operand 1 permutation test...")

perm_op1 = permutation_test_temporal(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=42,
    y=y_op1,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    n_shuffles=200,
    random_state=42
)

Running Operand 1 permutation test...


In [54]:
print("\nRunning Operand 2 permutation test...")

perm_op2 = permutation_test_temporal(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=42,
    y=y_op2,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    n_shuffles=200,
    random_state=43
)


Running Operand 2 permutation test...


In [55]:
print("\n========================================")
print("YFU.hpc.43 — OPERAND SPECIFICITY")
print("========================================")

for name, result in [
    ("Operand 1", perm_op1),
    ("Operand 2", perm_op2)
]:

    print(f"\n{name}")

    print(
        f"Real accuracy: "
        f"{100*result['real_accuracy']:.2f}%"
    )

    print(
        f"Best bin: "
        f"{result['best_bin']:.0f} ms"
    )

    print(
        f"Best Gamma: "
        f"{result['best_gamma']}"
    )

    print(
        f"Permutation null mean: "
        f"{100*result['null_mean']:.2f}%"
    )

    print(
        f"95th percentile of null: "
        f"{100*result['null_95']:.2f}%"
    )

    print(
        f"Permutation p-value: "
        f"{result['p_value']:.4f}"
    )


YFU.hpc.43 — OPERAND SPECIFICITY

Operand 1
Real accuracy: 11.48%
Best bin: 150 ms
Best Gamma: 0.2
Permutation null mean: 14.28%
95th percentile of null: 17.23%
Permutation p-value: 0.9552

Operand 2
Real accuracy: 14.00%
Best bin: 100 ms
Best Gamma: 0.2
Permutation null mean: 14.26%
95th percentile of null: 17.60%
Permutation p-value: 0.6020


No evidence of number coding in O1 or O2 separately


In [56]:
# ============================================================
# CHECK: Are there tied best models for Operand 2?
# ============================================================

grid_check, best_check = temporal_grid_search_one_neuron(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=42,
    y=y_op2,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    random_state=42
)

best_accuracy = grid_check["cv_accuracy"].max()

print("Maximum accuracy:")
print(f"{100 * best_accuracy:.4f}%")

print("\nAll parameter combinations achieving this maximum:")

tied_models = grid_check[
    np.isclose(
        grid_check["cv_accuracy"],
        best_accuracy
    )
][
    ["bin_ms", "gamma", "n_bins", "cv_accuracy"]
]

display(tied_models)

Maximum accuracy:
14.0000%

All parameter combinations achieving this maximum:


,bin_ms,gamma,n_bins,cv_accuracy
14,150,0.8,6,0.14


In [57]:
for run in range(2):

    _, best_repeat = temporal_grid_search_one_neuron(
        spike_matrix=yfu_spikes,
        presentations=yfu_op2,
        neuron_id=42,
        y=y_op2,
        candidate_bins=candidate_bins,
        candidate_gammas=candidate_gammas,
        random_state=42
    )

    print(
        f"Run {run+1}: "
        f"bin={best_repeat['bin_ms']}, "
        f"gamma={best_repeat['gamma']}, "
        f"accuracy={100*best_repeat['cv_accuracy']:.4f}%"
    )

Run 1: bin=150.0, gamma=0.8, accuracy=14.0000%
Run 2: bin=150.0, gamma=0.8, accuracy=14.0000%


In [58]:
# ============================================================
# PAPER-STYLE PERMUTATION TEST
# Fixed best bin size + Gamma during shuffles
# ============================================================

def permutation_test_temporal_fixed_params(
    spike_matrix,
    presentations,
    neuron_id,
    y,
    candidate_bins,
    candidate_gammas,
    n_shuffles=200,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    # --------------------------------------------------------
    # 1. Find best parameters from REAL labels
    # --------------------------------------------------------
    real_grid, real_best = temporal_grid_search_one_neuron(
        spike_matrix=spike_matrix,
        presentations=presentations,
        neuron_id=neuron_id,
        y=y,
        candidate_bins=candidate_bins,
        candidate_gammas=candidate_gammas,
        random_state=random_state
    )

    best_bin = int(real_best["bin_ms"])
    best_gamma = float(real_best["gamma"])
    real_accuracy = float(real_best["cv_accuracy"])

    # --------------------------------------------------------
    # 2. Build the temporal feature matrix ONCE
    #    using the selected real-data bin size
    # --------------------------------------------------------
    X = temporal_features_one_neuron(
        spike_matrix=spike_matrix,
        presentations=presentations,
        neuron_id=neuron_id,
        bin_ms=best_bin
    )

    classes, counts = np.unique(
        y,
        return_counts=True
    )

    n_folds = min(
        10,
        int(counts.min())
    )

    cv = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=random_state
    )

    priors = np.ones(
        len(classes)
    ) / len(classes)

    # --------------------------------------------------------
    # Helper: decode with FIXED Gamma
    # --------------------------------------------------------
    def decode_fixed(X, labels):

        y_true_all = []
        y_pred_all = []

        for train_idx, test_idx in cv.split(
            X,
            labels
        ):

            X_train = X[train_idx]
            X_test = X[test_idx]

            y_train = labels[train_idx]
            y_test = labels[test_idx]

            # Remove zero within-class variance features
            # using training data only
            keep = np.zeros(
                X_train.shape[1],
                dtype=bool
            )

            for j in range(X_train.shape[1]):

                variances = []

                for c in classes:

                    values = X_train[
                        y_train == c,
                        j
                    ]

                    variances.append(
                        np.var(values)
                    )

                keep[j] = np.any(
                    np.asarray(variances) > 0
                )

            if keep.sum() == 0:
                return np.nan

            lda = LinearDiscriminantAnalysis(
                solver="eigen",
                shrinkage=best_gamma,
                priors=priors
            )

            try:
                lda.fit(
                    X_train[:, keep],
                    y_train
                )

                pred = lda.predict(
                    X_test[:, keep]
                )

            except Exception:
                return np.nan

            y_true_all.extend(y_test)
            y_pred_all.extend(pred)

        return accuracy_score(
            y_true_all,
            y_pred_all
        )

    # --------------------------------------------------------
    # 3. Shuffles with parameters fixed
    # --------------------------------------------------------
    shuffled_accuracies = []

    for s in range(n_shuffles):

        y_shuffle = rng.permutation(y)

        acc_shuffle = decode_fixed(
            X,
            y_shuffle
        )

        if np.isfinite(acc_shuffle):
            shuffled_accuracies.append(
                acc_shuffle
            )

        if (s + 1) % 20 == 0:
            print(
                f"{s + 1}/{n_shuffles} shuffles",
                end="\r"
            )

    shuffled_accuracies = np.asarray(
        shuffled_accuracies
    )

    # --------------------------------------------------------
    # 4. Paper-style empirical p-value
    # --------------------------------------------------------
    p_value = (
        1
        + np.sum(
            shuffled_accuracies >= real_accuracy
        )
    ) / (
        1 + len(shuffled_accuracies)
    )

    return {
        "real_accuracy": real_accuracy,
        "best_bin": best_bin,
        "best_gamma": best_gamma,
        "p_value": p_value,
        "null_mean": shuffled_accuracies.mean(),
        "null_95": np.percentile(
            shuffled_accuracies,
            95
        ),
        "n_valid_shuffles":
            len(shuffled_accuracies)
    }

In [59]:
print("Operand 1...")

paper_perm_op1 = permutation_test_temporal_fixed_params(
    spike_matrix=yfu_spikes,
    presentations=yfu_op1,
    neuron_id=42,
    y=y_op1,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    n_shuffles=200,
    random_state=42
)

print("\nOperand 2...")

paper_perm_op2 = permutation_test_temporal_fixed_params(
    spike_matrix=yfu_spikes,
    presentations=yfu_op2,
    neuron_id=42,
    y=y_op2,
    candidate_bins=candidate_bins,
    candidate_gammas=candidate_gammas,
    n_shuffles=200,
    random_state=42
)

Operand 1...
200/200 shuffles
Operand 2...


In [60]:
print("\n========================================")
print("YFU.hpc.43 — PAPER-STYLE PERMUTATION")
print("========================================")

for name, result in [
    ("Operand 1", paper_perm_op1),
    ("Operand 2", paper_perm_op2)
]:

    print(f"\n{name}")
    print(
        f"Real accuracy: "
        f"{100*result['real_accuracy']:.2f}%"
    )
    print(
        f"Best bin: "
        f"{result['best_bin']} ms"
    )
    print(
        f"Best Gamma: "
        f"{result['best_gamma']}"
    )
    print(
        f"Shuffle mean: "
        f"{100*result['null_mean']:.2f}%"
    )
    print(
        f"Shuffle 95th percentile: "
        f"{100*result['null_95']:.2f}%"
    )
    print(
        f"Permutation p-value: "
        f"{result['p_value']:.4f}"
    )


YFU.hpc.43 — PAPER-STYLE PERMUTATION

Operand 1
Real accuracy: 11.48%
Best bin: 150 ms
Best Gamma: 0.2
Shuffle mean: 10.98%
Shuffle 95th percentile: 14.75%
Permutation p-value: 0.4428

Operand 2
Real accuracy: 14.00%
Best bin: 150 ms
Best Gamma: 0.8
Shuffle mean: 11.15%
Shuffle 95th percentile: 15.20%
Permutation p-value: 0.1244


it is still not significant at p<0.05 

In [61]:
# ============================================================
# OPERAND SPECIFICITY — STEP 5
# All 62 YFU MTL neurons
# ============================================================

operand_specific_results = []

n_neurons = len(yfu_mtl_indices)

for k, neuron_id in enumerate(yfu_mtl_indices, start=1):

    print(
        f"\nNeuron {k}/{n_neurons} "
        f"(unit {neuron_id + 1}, {yfu_regions[neuron_id]})"
    )

    # --------------------------------------------------------
    # Operand 1
    # --------------------------------------------------------

    op1_result = permutation_test_temporal_fixed_params(
        spike_matrix=yfu_spikes,
        presentations=yfu_op1,
        neuron_id=neuron_id,
        y=y_op1,
        candidate_bins=candidate_bins,
        candidate_gammas=candidate_gammas,
        n_shuffles=200,
        random_state=42
    )

    # --------------------------------------------------------
    # Operand 2
    # --------------------------------------------------------

    op2_result = permutation_test_temporal_fixed_params(
        spike_matrix=yfu_spikes,
        presentations=yfu_op2,
        neuron_id=neuron_id,
        y=y_op2,
        candidate_bins=candidate_bins,
        candidate_gammas=candidate_gammas,
        n_shuffles=200,
        random_state=42
    )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    operand_specific_results.append({

        "neuron_id": neuron_id,
        "paper_unit": neuron_id + 1,
        "region": yfu_regions[neuron_id],

        # Operand 1
        "op1_accuracy": op1_result["real_accuracy"],
        "op1_bin": op1_result["best_bin"],
        "op1_gamma": op1_result["best_gamma"],
        "op1_p": op1_result["p_value"],
        "op1_tuned": op1_result["p_value"] < 0.05,

        # Operand 2
        "op2_accuracy": op2_result["real_accuracy"],
        "op2_bin": op2_result["best_bin"],
        "op2_gamma": op2_result["best_gamma"],
        "op2_p": op2_result["p_value"],
        "op2_tuned": op2_result["p_value"] < 0.05
    })

    # --------------------------------------------------------
    # Save checkpoint after every neuron
    # --------------------------------------------------------

    temp_df = pd.DataFrame(
        operand_specific_results
    )

    temp_df.to_csv(
        tables_dir / "YFU_operand_specificity_TEMPORAL_checkpoint.csv",
        index=False
    )

    print(
        f"\nCompleted {k}/{n_neurons}"
    )

print("\nAll YFU neurons complete.")


Neuron 1/62 (unit 1, amy)
200/200 shuffles
Completed 1/62

Neuron 2/62 (unit 2, amy)
200/200 shuffles
Completed 2/62

Neuron 3/62 (unit 3, amy)
200/200 shuffles
Completed 3/62

Neuron 4/62 (unit 4, amy)
200/200 shuffles
Completed 4/62

Neuron 5/62 (unit 5, amy)
200/200 shuffles
Completed 5/62

Neuron 6/62 (unit 6, amy)
200/200 shuffles
Completed 6/62

Neuron 7/62 (unit 7, amy)
200/200 shuffles
Completed 7/62

Neuron 8/62 (unit 8, amy)
200/200 shuffles
Completed 8/62

Neuron 9/62 (unit 9, amy)
200/200 shuffles
Completed 9/62

Neuron 10/62 (unit 10, amy)
200/200 shuffles
Completed 10/62

Neuron 11/62 (unit 11, amy)
200/200 shuffles
Completed 11/62

Neuron 12/62 (unit 12, amy)
200/200 shuffles
Completed 12/62

Neuron 13/62 (unit 21, hpc)
200/200 shuffles
Completed 13/62

Neuron 14/62 (unit 22, hpc)
200/200 shuffles
Completed 14/62

Neuron 15/62 (unit 23, hpc)
200/200 shuffles
Completed 15/62

Neuron 16/62 (unit 24, hpc)
200/200 shuffles
Completed 16/62

Neuron 17/62 (unit 25, hpc)
200/20

In [62]:
yfu_operand_df = pd.DataFrame(
    operand_specific_results
)

def operand_category(row):

    if row["op1_tuned"] and row["op2_tuned"]:
        return "Both"

    elif row["op1_tuned"]:
        return "Operand 1 only"

    elif row["op2_tuned"]:
        return "Operand 2 only"

    else:
        return "Neither"


yfu_operand_df["category"] = (
    yfu_operand_df.apply(
        operand_category,
        axis=1
    )
)

In [63]:
print("================================")
print("YFU OPERAND-SPECIFIC TEMPORAL CODING")
print("================================")

print(
    yfu_operand_df["category"]
    .value_counts()
)

print("\nTotal neurons:", len(yfu_operand_df))

print(
    "Operand 1 significant:",
    yfu_operand_df["op1_tuned"].sum()
)

print(
    "Operand 2 significant:",
    yfu_operand_df["op2_tuned"].sum()
)

print(
    "Significant in either operand:",
    (
        yfu_operand_df["op1_tuned"]
        | yfu_operand_df["op2_tuned"]
    ).sum()
)

YFU OPERAND-SPECIFIC TEMPORAL CODING
category
Neither           29
Operand 1 only    14
Operand 2 only    12
Both               7
Name: count, dtype: int64

Total neurons: 62
Operand 1 significant: 21
Operand 2 significant: 19
Significant in either operand: 33


In [64]:
summary = (
    yfu_operand_df["category"]
    .value_counts()
    .reindex(
        [
            "Operand 1 only",
            "Operand 2 only",
            "Both",
            "Neither"
        ],
        fill_value=0
    )
)

summary_percent = (
    100 * summary / len(yfu_operand_df)
)

operand_summary = pd.DataFrame({
    "n_neurons": summary,
    "percent": summary_percent
})

display(operand_summary)

,n_neurons,percent
category,,
Operand 1 only,14,22.580645
Operand 2 only,12,19.354839
Both,7,11.290323
Neither,29,46.774194


In [65]:
from statsmodels.stats.multitest import multipletests

In [66]:
# ============================================================
# FDR CORRECTION — YFU OPERAND SPECIFICITY
# Benjamini-Hochberg, alpha = 0.05
# ============================================================

# Operand 1: 62 p-values
reject_op1, q_op1, _, _ = multipletests(
    yfu_operand_df["op1_p"].values,
    alpha=0.05,
    method="fdr_bh"
)

# Operand 2: 62 p-values
reject_op2, q_op2, _, _ = multipletests(
    yfu_operand_df["op2_p"].values,
    alpha=0.05,
    method="fdr_bh"
)

# Store results
yfu_operand_df["op1_q"] = q_op1
yfu_operand_df["op2_q"] = q_op2

yfu_operand_df["op1_fdr"] = reject_op1
yfu_operand_df["op2_fdr"] = reject_op2

In [67]:
def operand_category_fdr(row):

    if row["op1_fdr"] and row["op2_fdr"]:
        return "Both"

    elif row["op1_fdr"]:
        return "Operand 1 only"

    elif row["op2_fdr"]:
        return "Operand 2 only"

    else:
        return "Neither"


yfu_operand_df["category_fdr"] = (
    yfu_operand_df.apply(
        operand_category_fdr,
        axis=1
    )
)

In [68]:
print("======================================")
print("YFU MULTIPLE-COMPARISON CHECK")
print("======================================")

print("\nUNCORRECTED p < 0.05")
print(
    yfu_operand_df["category"]
    .value_counts()
)

print("\nFDR-CORRECTED q < 0.05")
print(
    yfu_operand_df["category_fdr"]
    .value_counts()
)

print("\nOperand 1")
print(
    "Uncorrected:",
    yfu_operand_df["op1_tuned"].sum()
)
print(
    "After FDR:",
    yfu_operand_df["op1_fdr"].sum()
)

print("\nOperand 2")
print(
    "Uncorrected:",
    yfu_operand_df["op2_tuned"].sum()
)
print(
    "After FDR:",
    yfu_operand_df["op2_fdr"].sum()
)

YFU MULTIPLE-COMPARISON CHECK

UNCORRECTED p < 0.05
category
Neither           29
Operand 1 only    14
Operand 2 only    12
Both               7
Name: count, dtype: int64

FDR-CORRECTED q < 0.05
category_fdr
Neither    62
Name: count, dtype: int64

Operand 1
Uncorrected: 21
After FDR: 0

Operand 2
Uncorrected: 19
After FDR: 0


In [71]:
554*0.05


27.700000000000003

In [69]:
display(
    yfu_operand_df[
        [
            "paper_unit",
            "region",
            "op1_p",
            "op1_q",
            "op1_fdr",
            "op2_p",
            "op2_q",
            "op2_fdr"
        ]
    ]
    .sort_values(
        ["op1_p", "op2_p"]
    )
    .head(20)
)

,paper_unit,region,op1_p,op1_q,op1_fdr,op2_p,op2_q,op2_fdr
25,34,hpc,0.004975,0.102819,False,0.004975,0.102819,False
22,31,hpc,0.004975,0.102819,False,0.134328,0.231343,False
46,55,hpc,0.004975,0.102819,False,0.243781,0.321310,False
55,64,hpc,0.009950,0.143947,False,0.248756,0.321310,False
30,39,hpc,0.014925,0.143947,False,0.039801,0.145157,False
45,54,amy,0.014925,0.143947,False,0.054726,0.161573,False
28,37,hpc,0.019900,0.143947,False,0.154229,0.239055,False
14,23,hpc,0.024876,0.143947,False,0.024876,0.145157,False
18,27,hpc,0.024876,0.143947,False,0.024876,0.145157,False
35,44,amy,0.029851,0.143947,False,0.109453,0.212065,False


In [70]:
yfu_operand_df.to_csv(
    tables_dir / "YFU_operand_specificity_TEMPORAL_with_FDR.csv",
    index=False
)



How many pooled temporal number-coding neurons survive FDR correction?}


In [72]:
# See existing dataframes and likely result variables

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        print(name, obj.shape, list(obj.columns))

RuntimeError: dictionary changed size during iteration